In [ ]:
import numpy as np
from scipy.linalg import orth

# Clear and initialize variables
m = 2048
n = 100
s = np.zeros(n)
Length = 10
Nst = n // Length

# Populate the vector 's' as per the given logic
for i in range(1, Nst + 1):
    s[Length * (i - 1):Length * i] = -0.6 * (i - 1)

# Handle the remaining elements
s[Length * Nst:] = -0.6 * (Nst - 1)

# Take 10^s element-wise
s = 10 ** s

# Create matrix M
M = orth(np.random.randn(n, n)) @ np.diag(s) @ orth(np.random.randn(n, n))

# Add rows of zeros to make the matrix size (m x n)
M = np.vstack([M, np.zeros((m - n, n))])

# Additional variables k and f
k = 50
f = 1

# Print the final matrix and variables (optional)
print("Matrix M:\n", M[:5,:5])
print("k:", k)
print("f:", f)
print(M.shape)

In [ ]:
import numpy as np

seed = 42  # Set the same seed value
np.random.seed(seed)  # Initialize the random number generator

# Generate a random matrix
A = np.random.rand(3,3).T
print(A)


In [ ]:
from partialQR import partialQR
import numpy as np
seed = 42  # Set the same seed value
np.random.seed(seed) 

M = np.random.rand(5, 5)
k = 3
Q, R, A, B, C = partialQR(M, k)

print(M)
print("Q :", Q)
print("R :", R)
print("A :", A)
print("B :", B)
print("C :", C)

In [ ]:
from maxn3 import maxn3

gamma, i, j = maxn3(M)

print("gamma :", gamma)
print("i :", i)
print("j :", j)

In [ ]:
from rrqr_k import rrqr_k
seed = 42  # Set the same seed value
np.random.seed(seed) 
M = np.random.rand(7, 7)
k = 3

Q, R, P, nb = rrqr_k(M, 3, 1)
print("Q : \n", Q)
print("R : \n", R)
print("P : \n", P)
print(nb)

In [ ]:
print("\n", Q*R)

In [ ]:
print(M)

In [ ]:
from Kahan import kahan
A = kahan(6)
print("A \n", A)

In [ ]:
import numpy as np

def strong_RRQR(M, f, delta):
    """
    Perform a Strong RRQR factorization on matrix M.
    
    Parameters:
    M (numpy array): The input matrix for factorization.
    f (float): The tolerance factor used in the factorization process.
    
    Returns:
    tuple: A tuple (Q, R, P) representing the QR factorization, where Q is orthogonal, 
    R is upper triangular, and P is a permutation matrix.
    """
    def initialize(M):
        """
        Initialize matrices and vectors for the RRQR factorization.
        """
        # Initial values
        k = 0
        R = M.copy()
        Ck = M.copy()
        Phi = np.eye(M.shape[1])  # Identity matrix representing permutation
        return k, R, Ck, Phi

    def update_matrices(A_k, B_k C_k, k, jmax, Phi, gamma_star, omega_star):
        """
        Update the matrices based on the selected column for permutation.
        """
        gamma_1 = gamma_star[0]
        # Permutation matrix update
        Phi[:, [k, k + jmax]] = Phi[:, [k + jmax, k]]
        # Compute the updated R and Ck after swapping
        R[:, [k, k + jmax]] = R[:, [k + jmax, k]]
        Ck = R[k:, k:]
        return R, Ck, Phi

    def compute_rows_norm(A):
        """
        Compute the row norms of a matrix A.
        """
        return np.linalg.norm(A, axis=0)[:, np.newaxis]
    
    def compute_columns_norm(A):
        """
        Compute the row norms of a matrix A.
        """
        return np.linalg.norm(A, axis=1)[np.newaxis, :]
    
    def compute_rho_and_indices(A, B):
        max_matrix = np.maximum(abs_of_A_inv_B, ratio_matrix)

        # Step 2: Find the maximum value in the max_matrix
        max_value = np.max(max_matrix)
        indices = np.unravel_index(np.argmax(max_matrix), max_matrix.shape)
        
        rho = max_value
        
        return rho, indices
        

    # Step 1: Initialization
    k, R, Ck, Phi = initialize(M)
    
    # Compute initial norms for comparison
    A_k = R[:k, :k]
    B_k = R[:k, k:]
    C_k = R[k:, k:]
    
    omega_star = compute_rows_norm(A_k)
    gamma_star = compute_columns_norm(Ck)

    A_inv_B = np.linalg.inv(A_k) @ B_k
    
    while np.max(gamma_star) >= delta:
        # Step 2: Find jmax
        jmax = np.argmax(gamma_star)  # Index of maximum y_j(Ck)
        
        # Step 3: Update matrices
        k += 1
        R, Ck, Phi = update_matrices(R, Ck, k, jmax, Phi) # NEED TO BE CHANGED
        
        # Step 4: Recalculate norms
        omega_star = compute_rows_norm(A_k)
        gamma_star = compute_columns_norm(Ck)

        # Compute the ratio gamma_j / omega_i, resulting in a (3, 5) matrix
        ratio_matrix = gamma_star_reshaped / omega_star_reshaped
        
        abs_of_A_inv_B = np.abs(A_inv_B)
        
        rho_hat, [i, j] = compute_rho_and_indices(abs_of_A_inv_B, ratio_matrix)
        # Inner loop: Adjustments based on tolerance
        while rho_hat > f: # NEED TO BE CHANGED
            # Find appropriate i, j and update again
            R, Ck, Phi = update_matrices(R, Ck, i, j, Phi) # NEED TO BE CHANGED
            
            omega_star = compute_rows_norm(A_k)
            gamma_star = compute_columns_norm(Ck)
            ratio_matrix = gamma_star_reshaped / omega_star_reshaped
            
            A_k = R[:k, :k]
            B_k = R[:k, k:]
            C_k = R[k:, k:]
            
            A_inv_B = np.linalg.inv(A_k) @ B_k
        
            abs_of_A_inv_B = np.abs(A_inv_B)
            rho_hat, [i, j] = compute_rho_and_indices(abs_of_A_inv_B, ratio_matrix)

    return R, Phi

# Example usage
M = np.random.rand(5, 5)
f = 2
R, Phi = strong_RRQR(M, f)
print("R:", R)
print("Permutation matrix Phi:", Phi)


In [ ]:
M = np.random.rand(3, 5)
M

In [ ]:
omega_star = np.linalg.norm(M, axis=1) # norm of the columns
omega_star

 omega_star = compute_rows_norm(A_k)
        
        
        omega_star_reshaped = omega_star[:, np.newaxis]  # Shape (3, 1)
        gamma_star_reshaped = gamma_star[np.newaxis, :]

In [ ]:
def compute_columns_norm(A):
        """
        Compute the row norms of a matrix A.
        """
        return np.linalg.norm(A, axis=1)[np.newaxis, :]
    
gamma_star = compute_columns_norm(M)
gamma_star.shape

In [ ]:
# Reshape omega_star to be (3, 1) and gamma_star to be (1, 5) for broadcasting
omega_star_reshaped = omega_star[:, np.newaxis]  # Shape (3, 1)
gamma_star_reshaped = gamma_star[np.newaxis, :]  # Shape (1, 5)

# Compute the ratio gamma_j / omega_i, resulting in a (3, 5) matrix
ratio_matrix = gamma_star_reshaped / omega_star_reshaped

print("M:", M)
print("gamma_star:", gamma_star)
print("omega_star:", omega_star)
print("gamma_star / omega_star (matrix):\n", ratio_matrix)

In [ ]:
import numpy as np

# Example matrices A and B
A = np.array([[1, 5, 3], [7, 2, 4]])
B = np.array([[4, 3, 6], [1, 8, 2]])

# Step 1: Compute the element-wise maximum between A and B
max_matrix = np.maximum(A, B)

# Step 2: Find the maximum value in the max_matrix
max_value = np.max(max_matrix)

# Step 3: Find the indices of the maximum value
indices = np.unravel_index(np.argmax(max_matrix), max_matrix.shape)

# Print the results
print("Matrix A:\n", A)
print("Matrix B:\n", B)
print("Element-wise maximum matrix:\n", max_matrix)
print("Maximum value:", max_value)
print("Indices of maximum value:", indices)


In [ ]:
A = np.array([[-11, 5, 3], [7, 2, 4]])
A

In [ ]:
np.abs(A)

In [ ]:
import numpy as np
from scipy.linalg import qr, solve_triangular, inv

'''
def givens(a, b):
    """Create Givens rotation matrix G to zero out the element b."""
    r = np.hypot(a, b)
    c = a / r
    s = -b / r
    return np.array([[c, -s], [s, c]])
'''

def sRRQR_tol(A, f, tol):
    """
    Strong Rank Revealing QR with an error threshold tol
    Args:
        A (ndarray): Target matrix to be approximated.
        f (float): Constant that bounds the entries of inv(R11) * R12.
        tol (float): Error tolerance for truncating the decomposition.
    Returns:
        Q (ndarray): Orthogonal matrix Q from QR decomposition.
        R (ndarray): Upper triangular matrix from QR decomposition.
        p (ndarray): Column pivot indices.
    """

    # Check the constant bound f
    if f <= 1:
        print('Parameter f is less or equal to 1. Automatically setting f = 2.')
        f = 2

    # Get dimensions of A
    m, n = A.shape

    # Perform pivoted QR decomposition
    Q, R, p = qr(A, pivoting=True, mode='economic')

    # Make diagonals of R positive
    ss = np.sign(np.diag(R)) if min(R.shape) > 1 else np.sign(R[0, 0])
    R = R * ss[:, np.newaxis]
    Q = Q * ss

    # Determine rank k where diag(R) > tol
    diagR = np.diag(R)
    k = np.argmax(diagR <= tol)
    k = n if k == 0 else k

    # Special case: each column norm is less than tol
    if k == 0:
        return np.zeros((m, 0)), np.zeros((0, n)), np.arange(n)

    # Special case: rank k == n
    if k == n:
        print('Rank equals the number of columns!')
        return Q, R, p

    # Initialization of A^{-1}B (A refers to R11, B refers to R12)
    AB = solve_triangular(R[:k, :k], R[:k, k:], lower=False)

    # Initialization of gamma (norm of C's columns)
    gamma = np.linalg.norm(R[k:, k:], axis=0) if k < R.shape[0] else np.zeros(n - k)

    # Initialization of omega (reciprocal of inv(A)'s row norm)
    tmp = solve_triangular(R[:k, :k], np.eye(k), lower=False)
    omega = np.linalg.norm(tmp, axis=1)**(-1)

    # Main loop
    while True:
        # Step 1: Strong RRQR with fixed rank k
        while True:
            tmp = (1 / omega[:, None] @ gamma[None, :])**2 + AB**2
            res = np.where(tmp > f*f)
            if len(res[0]) == 0:
                break
            i, j = res[0][0], res[1][0]

            # Swap columns k+j and k+1, and update matrices
            if j > 0:
                AB[:, [0, j]] = AB[:, [j, 0]]
                gamma[[0, j]] = gamma[[j, 0]]
                R[:, [k, k+j]] = R[:, [k+j, k]]
                p[[k, k+j]] = p[[k+j, k]]

            if i < k:
                # Interchanging the i-th and k-th columns
                p[i:k+1] = np.roll(p[i:k+1], shift=-1)
                R[:, i:k+1] = np.roll(R[:, i:k+1], shift=-1, axis=1)
                omega[i:k+1] = np.roll(omega[i:k+1], shift=-1)
                AB[i:k+1, :] = np.roll(AB[i:k+1, :], shift=-1, axis=0)

                # Givens rotation for triangulation of R(1:k, 1:k)
                for ii in range(i, k):
                    G = givens(R[ii, ii], R[ii+1, ii])
                    R[ii:ii+2, :] = G @ R[ii:ii+2, :]
                    Q[:, ii:ii+2] = Q[:, ii:ii+2] @ G.T

                if R[k, k] < 0:
                    R[k, :] = -R[k, :]
                    Q[:, k] = -Q[:, k]

            # Zeroing out the below-diagonal elements of R(k+1, k+1)
            if k < R.shape[0] - 1:
                for ii in range(k+2, R.shape[0]):
                    G = givens(R[k+1, k+1], R[ii, k+1])
                    R[[k+1, ii], :] = G @ R[[k+1, ii], :]
                    Q[:, [k+1, ii]] = Q[:, [k+1, ii]] @ G.T

        # Step 2: Check if we can reduce rank (k -> k-1)
        min_omega, i = np.min(omega), np.argmin(omega)
        if min_omega > tol:
            break

        # Interchanging the i-th and k-th columns
        if i < k:
            p[i:k+1] = np.roll(p[i:k+1], shift=-1)
            R[:, i:k+1] = np.roll(R[:, i:k+1], shift=-1, axis=1)

            # Givens rotation for triangulation
            for ii in range(i, k):
                G = givens(R[ii, ii], R[ii+1, ii])
                R[ii:ii+2, :] = G @ R[ii:ii+2, :]
                Q[:, ii:ii+2] = Q[:, ii:ii+2] @ G.T

            if R[k, k] < 0:
                R[k, :] = -R[k, :]
                Q[:, k] = -Q[:, k]

        k -= 1

        # Update A^{-1}B, omega, and gamma
        AB = solve_triangular(R[:k, :k], R[:k, k:], lower=False)
        gamma = np.linalg.norm(R[k:, k:], axis=0) if k < R.shape[0] else np.zeros(n - k)
        tmp = solve_triangular(R[:k, :k], np.eye(k), lower=False)
        omega = np.linalg.norm(tmp, axis=1)**(-1)

    # Return truncated version of the strong RRQR decomposition
    return Q[:, :k], R[:k, :], p

In [1]:
import numpy as np
from scipy.linalg import qr

def givens(x, y):
    if y == 0:
        c = 1
        s = 0
    else:
        if abs(y) >= abs(x):
            cotangent = x / y
            s = 1 / np.sqrt(1 + cotangent**2)
            c = s * cotangent
        else:
            tangent = y / x
            c = 1 / np.sqrt(1 + tangent**2)
            s = c * tangent

    return np.array([[c, s], [-s, c]])

import numpy as np
from scipy.linalg import qr, solve_triangular

def sRRQR_tol2(A, f, tol):
    """
    Strong Rank Revealing QR with an error threshold tol
        A[:, p] = Q * R = Q [R11, R12;
                               0, R22]
    where R11 and R12 satisfies that matrix (inv(R11) * R12) has entries
    bounded by a pre-specified constant which should be not less than 1.

    Input:
        A, matrix,  target matrix that is approximated.
        f, scalar,  constant that bound the entries of calculated (inv(R11) * R12)
        tol, scalar, error threshold.

    Output:
        A[:, p] = [Q1, Q2] * [R11, R12;
                               0, R22]
               approx Q1 * [R11, R12];
        Only truncated QR decomposition is returned as
            Q = Q1,
            R = [R11, R12];
        where each column of R22 is less than tol.

    Reference:
        Gu, Ming, and Stanley C. Eisenstat. "Efficient algorithms for
        computing a strong rank-revealing QR factorization." SIAM Journal
        on Scientific Computing 17.4 (1996): 848-869.

    Note:
        Algorithm 5 in the above ref. is implemented.
    """

    # check constant bound f
    if f < 1:
        print('parameter f given is less than 1. Automatically set f = 2')
        f = 2

    # dimension of the given matrix
    m, n = A.shape

    # pivoting QR first (generally most time consuming step)
    Q, R, p = qr(A, pivoting=True)

    # make diagonals of R positive.
    if R.shape[0] == 1 or R.shape[1] == 1:
        ss = np.sign(R[0, 0])
    else:
        ss = np.sign(np.diag(R))
    R = R * ss[:, np.newaxis]
    Q = Q * ss

    # check a sufficiently large rank k to meet the threshold 'tol'
    k = None
    if R.shape[0] == 1 or R.shape[1] == 1:
        if R[0, 0] > tol:
            k = 1
    else:
        k = np.where(np.diag(R) > tol)[0][-1] + 1

    # special case : each column has norm less than the threshold.
    if k is None:
        Q = np.zeros((m, 0))
        R = np.zeros((0, n))
        p = np.arange(n)
        return Q, R, p

    # special case : rank k == n
    if k == n:
        print('Rank equals the number of columns!')
        return Q, R, p

    # The following codes are the major part of the strong rank-revealing
    # algorithm which is based on the above pivoting QR.
    # Name of variables are from the reference paper.

    # Initialization of A^{-1}B ( A refers to R11, B refers to R12)
    AB = solve_triangular(R[:k, :k], R[:k, k:], lower=False)

    # Initialization of gamma, i.e., norm of C's columns (C refers to R22)
    gamma = np.zeros(n - k)
    if k != R.shape[0]:
        gamma = np.sqrt(np.sum(R[k:, k:] ** 2, axis=0))

    # Initialization of omega, i.e., reciprocal of inv(A)'s row norm
    tmp = solve_triangular(R[:k, :k], np.eye(k), lower=False)
    omega = 1 / np.sqrt(np.sum(tmp ** 2, axis=1))

    # KEY STEP in Strong RRQR:
    # "while" loop for interchanging the columns from first k columns and
    # the remaining (n-k) columns.

    # STEP 1: Strong RRQR with the fixed rank k.
    # STEP 2: Check whether a smaller rank k can meet the threshold.

    Rm = R.shape[0]

    while True:
        # STEP 1: Strong RRQR with the fixed rank k.
        while True:
            # identify interchanging columns
            tmp = (1 / omega[:, np.newaxis] * gamma) ** 2 + AB ** 2
            i, j = np.unravel_index(np.argmax(tmp > f * f), tmp.shape)

            # if no entry of tmp greater than f*f, no interchange needed and the
            # present Q, R, p is a strong RRQR.
            if tmp[i, j] <= f * f:
                break

            # Interchange the i th and (k+j) th column of target matrix A and
            # update QR decomposition (Q, R, p), AB, gamma, and omega.
            # First step : interchanging the k+1 and k+j th columns
            if j > 0:
                AB[:, [0, j]] = AB[:, [j, 0]]
                gamma[[0, j]] = gamma[[j, 0]]
                R[:, [k, k + j]] = R[:, [k + j, k]]
                p[[k, k + j]] = p[[k + j, k]]

            # Second step : interchanging the i and k th columns
            if i < k:
                p[i:k + 1] = np.roll(p[i:k + 1], -1)
                R[:, i:k + 1] = np.roll(R[:, i:k + 1], -1, axis=1)
                omega[i:k + 1] = np.roll(omega[i:k + 1], -1)
                AB[i:k + 1, :] = np.roll(AB[i:k + 1, :], -1, axis=0)
                # givens rotation for the triangulation of R(1:k, 1:k)
                for ii in range(i, k):
                    G = givens_rotation(R[ii, ii], R[ii + 1, ii])
                    if G[0, :] @ np.array([R[ii, ii], R[ii + 1, ii]]) < 0:
                        G = -G  # guarantee R(ii,ii) > 0
                    R[ii:ii + 2, :] = G @ R[ii:ii + 2, :]
                    Q[:, ii:ii + 2] = Q[:, ii:ii + 2] @ G.T
                if R[k, k] < 0:
                    R[k, :] = -R[k, :]
                    Q[:, k] = -Q[:, k]

            # Third step : zeroing out the below-diag of k+1 th columns
            if k < Rm:
                for ii in range(k + 2, Rm):
                    G = givens_rotation(R[k + 1, k + 1], R[ii, k + 1])
                    if G[0, :] @ np.array([R[k + 1, k + 1], R[ii, k + 1]]) < 0:
                        G = -G  # guarantee R(k+1,k+1) > 0
                    R[[k + 1, ii], :] = G @ R[[k + 1, ii], :]
                    Q[:, [k + 1, ii]] = Q[:, [k + 1, ii]] @ G.T

            # Fourth step : interchanging the k and k+1 th columns
            p[[k, k + 1]] = p[[k + 1, k]]
            ga = R[k, k]
            mu = R[k, k + 1] / ga
            if k < Rm:
                nu = R[k + 1, k + 1] / ga
            else:
                nu = 0
            rho = np.sqrt(mu * mu + nu * nu)
            ga_bar = ga * rho
            b1 = R[:k - 1, k]
            b2 = R[:k - 1, k + 1]
            c1T = R[k, k + 2:]
            if k + 1 > Rm:
                c2T = np.zeros_like(c1T)
            else:
                c2T = R[k + 1, k + 2:]
            c1T_bar = (mu * c1T + nu * c2T) / rho
            c2T_bar = (nu * c1T - mu * c2T) / rho

            # modify R
            R[:k - 1, k] = b2
            R[:k - 1, k + 1] = b1
            R[k, k] = ga_bar
            R[k, k + 1] = ga * mu / rho
            R[k + 1, k + 1] = ga * nu / rho
            R[k, k + 2:] = c1T_bar
            R[k + 1, k + 2:] = c2T_bar

            # update AB
            u = solve_triangular(R[:k - 1, :k - 1], b1, lower=False)
            u1 = AB[:k - 1, 0]
            AB[:k - 1, 0] = (nu * nu * u - mu * u1) / rho / rho
            AB[k, 0] = mu / rho / rho
            AB[k, 1:] = c1T_bar / ga_bar
            AB[:k - 1, 1:] = AB[:k - 1, 1:] + (nu * u * c2T_bar - u1 * c1T_bar) / ga_bar

            # update gamma
            gamma[0] = ga * nu / rho
            gamma[1:] = np.sqrt(gamma[1:] ** 2 + (c2T_bar ** 2) - (c2T ** 2))

            # update omega
            u_bar = u1 + mu * u
            omega[k] = ga_bar
            omega[:k - 1] = (omega[:k - 1] ** (-2) + u_bar ** 2 / (ga_bar * ga_bar) - u ** 2 / (ga * ga)) ** (-1 / 2)

            # Eliminate new R(k+1, k) by orthogonal transformation
            Gk = np.array([[mu / rho, nu / rho], [nu / rho, -mu / rho]])
            if k < Rm:
                Q[:, [k, k + 1]] = Q[:, [k, k + 1]] @ Gk.T

        # STEP 2: Check whether k -> k-1 can meet the threshold.
        tmp, i = min(omega), np.argmin(omega)
        # rank (k-1) doesn't meet the threshold, stop
        if tmp > tol:
            break

        # interchanging the i and k th columns
        if i < k:
            p[i:k + 1] = np.roll(p[i:k + 1], -1)
            R[:, i:k + 1] = np.roll(R[:, i:k + 1], -1, axis=1)
            # givens rotation for the triangulation of R(1:k, 1:k)
            for ii in range(i, k):
                G = givens_rotation(R[ii, ii], R[ii + 1, ii])
                if G[0, :] @ np.array([R[ii, ii], R[ii + 1, ii]]) < 0:
                    G = -G
                R[ii:ii + 2, :] = G @ R[ii:ii + 2, :]
                Q[:, ii:ii + 2] = Q[:, ii:ii + 2] @ G.T
            if R[k, k] < 0:
                R[k, :] = -R[k, :]
                Q[:, k] = -Q[:, k]
        k -= 1

        # Update A^{-1}B, omega, gamma (can be improved)
        AB = solve_triangular(R[:k, :k], R[:k, k:], lower=False)  # A^{-1} * B
        gamma = np.sqrt(np.sum(R[k:, k:] ** 2, axis=0))
        tmp = solve_triangular(R[:k, :k], np.eye(k), lower=False)
        omega = 1 / np.sqrt(np.sum(tmp ** 2, axis=1))

    # Only return the truncated version of the strong RRQR decomposition
    Q = Q[:, :k]
    R = R[:k, :]
    return Q, R, p


In [2]:
import numpy as np
from scipy.spatial.distance import cdist
from scipy.linalg import norm

# Function sRRQR would be the Python translation of the MATLAB sRRQR you shared before.
np.random.seed(seed=42)
# Generate two well-separated point clusters
X0 = (4 * (1 - 2 * np.random.rand(3, 2000).T))
print(X0.shape)
print("X0:\n", X0)
np.random.seed(seed=42)
Y0 = 4 * (1 - 2 * np.random.rand(3, 2000).T)
# Add 12 only to the first column
Y0[:, 0] += 12

print(Y0.shape)
print("Y0:\n", Y0)

# Kernel block defined by K(x, y) = 1 / |x - y|
dist = cdist(X0, Y0)
A = 1.0 / dist
print("A:\n", A)

# Strong Rank Revealing QR with given error threshold
tol = 1e-5
f = 2

# Call the sRRQR function (assuming the function sRRQR_tol has been translated to Python)
Q, R, p = sRRQR_tol2(A, f, tol)

# Approximation error
error = norm(A[:, p] - Q @ R, 'fro') / norm(A, 'fro')

# Absolute maximum of entries in inv(R11)*R12
k = R.shape[0]
tmp = np.linalg.solve(R[:k, :k], R[:k, k:])
entry = np.max(np.abs(tmp))

# Print result
print(f'Approximation rank: {k}')
print(f'Relative approx. error: {error:.3E}')
print(f'Maximum entry in inv(R11)*R12: {entry:.3f}')

(2000, 3)
X0:
 [[ 1.00367905  1.90635453 -0.57596703]
 [-3.60571445  2.02416961 -2.44345863]
 [-1.85595153 -3.25003664 -2.08128744]
 ...
 [ 3.44833587  0.84342148 -2.1211908 ]
 [ 3.54356223 -0.2395247   0.11767493]
 [ 1.7425034   2.70906113  2.80494691]]
(2000, 3)
Y0:
 [[13.00367905  1.90635453 -0.57596703]
 [ 8.39428555  2.02416961 -2.44345863]
 [10.14404847 -3.25003664 -2.08128744]
 ...
 [15.44833587  0.84342148 -2.1211908 ]
 [15.54356223 -0.2395247   0.11767493]
 [13.7425034   2.70906113  2.80494691]]
A:
 [[0.08333333 0.13116802 0.09432254 ... 0.06865346 0.06796369 0.07573312]
 [0.05982842 0.08333333 0.06788374 ... 0.05237434 0.05140872 0.05513384]
 [0.06328823 0.08670585 0.08333333 ... 0.05623693 0.05619736 0.05747716]
 ...
 [0.10269402 0.1962655  0.12742156 ... 0.08333333 0.080983   0.0864781 ]
 [0.10282552 0.16851829 0.13191621 ... 0.0822248  0.08333333 0.09131221]
 [0.08485278 0.11763807 0.08771522 ... 0.0681049  0.06960832 0.08333333]]
Approximation rank: 80
Relative approx. er

In [ ]:
import numpy as np
from scipy.linalg import qr, solve_triangular, inv

'''
def givens(a, b):
    """Create Givens rotation matrix G to zero out the element b."""
    r = np.hypot(a, b)
    c = a / r
    s = -b / r
    return np.array([[c, -s], [s, c]])
'''

def sRRQR_Algo5(A, f, tol):
    """
    Strong Rank Revealing QR with an error threshold tol
    Args:
        A (ndarray): Target matrix to be approximated.
        f (float): Constant that bounds the entries of inv(R11) * R12.
        tol (float): Error tolerance for truncating the decomposition.
    Returns:
        Q (ndarray): Orthogonal matrix Q from QR decomposition.
        R (ndarray): Upper triangular matrix from QR decomposition.
        p (ndarray): Column pivot indices.
    """

    # Check the constant bound f
    if f < 1:
        print('Parameter f is less or equal to 1. Automatically setting f = 2.')
        f = 2

    # Get dimensions of A
    m, n = A.shape

    # Perform pivoted QR decomposition
    Q, R, p = qr(A, pivoting=True, mode='economic')

    # Make diagonals of R positive
    ss = np.sign(np.diag(R)) if min(R.shape) > 1 else np.sign(R[0, 0])
    R = R * ss[:, np.newaxis]
    Q = Q * ss

    # Determine rank k where diag(R) > tol
    diagR = np.diag(R)
    k = np.argmax(diagR <= tol)

    k = 0

    # Initialization of A^{-1}B (A refers to R11, B refers to R12)
    AB = solve_triangular(R[:k, :k], R[:k, k:], lower=False)

    # Initialization of gamma (norm of C's columns)
    gamma = np.linalg.norm(R[k:, k:], axis=0) if k < R.shape[0] else np.zeros(n - k)

    # Initialization of omega (reciprocal of inv(A)'s row norm)
    tmp = solve_triangular(R[:k, :k], np.eye(k), lower=False)
    omega = np.linalg.norm(tmp, axis=1)**(-1)

    # Main loop
    while True:
        # Step 1: Strong RRQR with fixed rank k
        while True:
            tmp = (1 / omega[:, None] @ gamma[None, :])**2 + AB**2
            res = np.where(tmp > f*f)
            if len(res[0]) == 0:
                break
            i, j = res[0][0], res[1][0]

            # Swap columns k+j and k+1, and update matrices
            if j > 0:
                AB[:, [0, j]] = AB[:, [j, 0]]
                gamma[[0, j]] = gamma[[j, 0]]
                R[:, [k, k+j]] = R[:, [k+j, k]]
                p[[k, k+j]] = p[[k+j, k]]

            if i < k:
                # Interchanging the i-th and k-th columns
                p[i:k+1] = np.roll(p[i:k+1], shift=-1)
                R[:, i:k+1] = np.roll(R[:, i:k+1], shift=-1, axis=1)
                omega[i:k+1] = np.roll(omega[i:k+1], shift=-1)
                AB[i:k+1, :] = np.roll(AB[i:k+1, :], shift=-1, axis=0)

                # Givens rotation for triangulation of R(1:k, 1:k)
                for ii in range(i, k):
                    G = givens(R[ii, ii], R[ii+1, ii])
                    R[ii:ii+2, :] = G @ R[ii:ii+2, :]
                    Q[:, ii:ii+2] = Q[:, ii:ii+2] @ G.T

                if R[k, k] < 0:
                    R[k, :] = -R[k, :]
                    Q[:, k] = -Q[:, k]

            # Zeroing out the below-diagonal elements of R(k+1, k+1)
            if k < R.shape[0] - 1:
                for ii in range(k+2, R.shape[0]):
                    G = givens(R[k+1, k+1], R[ii, k+1])
                    R[[k+1, ii], :] = G @ R[[k+1, ii], :]
                    Q[:, [k+1, ii]] = Q[:, [k+1, ii]] @ G.T

        # Step 2: Check if we can reduce rank (k -> k-1)
        min_omega, i = np.min(omega), np.argmin(omega)
        if min_omega > tol:
            break

        # Interchanging the i-th and k-th columns
        if i < k:
            p[i:k+1] = np.roll(p[i:k+1], shift=-1)
            R[:, i:k+1] = np.roll(R[:, i:k+1], shift=-1, axis=1)

            # Givens rotation for triangulation
            for ii in range(i, k):
                G = givens(R[ii, ii], R[ii+1, ii])
                R[ii:ii+2, :] = G @ R[ii:ii+2, :]
                Q[:, ii:ii+2] = Q[:, ii:ii+2] @ G.T

            if R[k, k] < 0:
                R[k, :] = -R[k, :]
                Q[:, k] = -Q[:, k]

        k -= 1

        # Update A^{-1}B, omega, and gamma
        AB = solve_triangular(R[:k, :k], R[:k, k:], lower=False)
        gamma = np.linalg.norm(R[k:, k:], axis=0) if k < R.shape[0] else np.zeros(n - k)
        tmp = solve_triangular(R[:k, :k], np.eye(k), lower=False)
        omega = np.linalg.norm(tmp, axis=1)**(-1)

    # Return truncated version of the strong RRQR decomposition
    return Q[:, :k], R[:k, :], p

In [ ]:
import numpy as np
from partialQR import partialQR
from maxn3 import maxn3

def strong_RRQR_Algo5(M, f, delta):
    """
    Perform a Strong RRQR factorization on matrix M.
    
    Parameters:
    M (numpy array): The input matrix for factorization.
    f (float): The tolerance factor used in the factorization process.
    
    Returns:
    tuple: A tuple (Q, R, P) representing the QR factorization, where Q is orthogonal, 
    R is upper triangular, and P is a permutation matrix.
    """
    def initialize(M):
        """
        Initialize matrices and vectors for the RRQR factorization.
        """
        # Initial values
        k = 0
        R = M.copy()
        Ck = M.copy()
        Phi = np.eye(M.shape[1])  # Identity matrix representing permutation
        return k, R, Ck, Phi

    def update_matrices(Ak, Bk Ck, k, jmax, Phi, gamma_star, omega_star):
        """
        Update the matrices based on the selected column for permutation.
        """
        b = Bk[:, 0]
        B = Bk[:, 1:]

        gamma = 
        c = 
        C = 

        Ak = 
        Bk =
        Ck = 

        A_inv_B = 

        
        gamma_1 = gamma_star[0]
        # Permutation matrix update
        Phi[:, [k, k + jmax]] = Phi[:, [k + jmax, k]]
        # Compute the updated R and Ck after swapping
        R[:, [k, k + jmax]] = R[:, [k + jmax, k]]
        Ck = R[k:, k:]
        return R, Ck, Phi

    def compute_rows_norm(A):
        """
        Compute the row norms of a matrix A.
        """
        return np.linalg.norm(A, axis=0)[:, np.newaxis]
    
    def compute_columns_norm(A):
        """
        Compute the row norms of a matrix A.
        """
        return np.linalg.norm(A, axis=1)[np.newaxis, :]
    
    def compute_rho_and_indices(A, B):
        max_matrix = np.maximum(A, B)

        # Step 2: Find the maximum value in the max_matrix
        max_value = np.max(max_matrix)
        indices = np.unravel_index(np.argmax(max_matrix), max_matrix.shape)
        
        rho = max_value
        
        return rho, indices
        

    # Step 1: Initialization
    k, R, Ck, Phi = initialize(M)
    
    # Compute initial norms for comparison
    A_k = R[:k, :k]
    B_k = R[:k, k:]
    C_k = R[k:, k:]
    
    omega_star = compute_rows_norm(A_k)
    gamma_star = compute_columns_norm(Ck)

    A_inv_B = np.linalg.inv(A_k) @ B_k

    #solve_triangular(R[:k, :k], R[:k, k:], lower=False)
    
    while np.max(gamma_star) >= delta:
        # Step 2: Find jmax
        jmax = np.argmax(gamma_star)  # Index of maximum y_j(Ck)
        
        # Step 3: Update matrices
        k += 1
        # Update everything 

        _, n = M.shape
        _, R, A, B, C = partialQR(M, k)
    
        P = np.eye(n)
        invA = np.linalg.inv(A)
        nb = 0

        # Compute norms for gamma and omega
        gamma_lst = np.linalg.norm(C, axis=0)
        gamma, jmax = np.max(gamma_lst), np.argmax(gamma_lst)

        omega_lst = np.linalg.norm(invA, axis=1)
        omega, imax = np.max(omega_lst), np.argmax(omega_lst)
    
            

        rou = gamma * omega

        # Check if rou is less than f
        if rou <= f:
            rou, imax, jmax = maxn3(np.linalg.solve(A, B))

        if rou <= f:
            break

        # Swap the columns of P and R
        P[:, [jmax + k, imax]] = P[:, [imax, jmax + k]]
        R[:, [jmax + k, imax]] = R[:, [imax, jmax + k]]

        # Recompute the QR factorization
        _, R, A, B, C = partialQR(R, k)
        invA = np.linalg.inv(A)
        nb += 1

    Q = np.dot(M, np.linalg.inv(R))

    return Q, R, P, nb




        
        R, Ck, Phi = update_matrices(R, Ck, k, jmax, Phi) # NEED TO BE CHANGED
        
        # Step 4: Recalculate norms
        omega_star = compute_rows_norm(A_k)
        gamma_star = compute_columns_norm(Ck)

        # Compute the ratio gamma_j / omega_i, resulting in a (3, 5) matrix
        ratio_matrix = gamma_star_reshaped / omega_star_reshaped
        
        abs_of_A_inv_B = np.abs(A_inv_B)
        
        rho_hat, [i, j] = compute_rho_and_indices(abs_of_A_inv_B, ratio_matrix)
        # Inner loop: Adjustments based on tolerance
        while rho_hat > f: # NEED TO BE CHANGED
            # Find appropriate i, j and update again
            R, Ck, Phi = update_matrices(R, Ck, i, j, Phi) # NEED TO BE CHANGED
            
            omega_star = compute_rows_norm(A_k)
            gamma_star = compute_columns_norm(Ck)
            ratio_matrix = gamma_star_reshaped / omega_star_reshaped
            
            A_k = R[:k, :k]
            B_k = R[:k, k:]
            C_k = R[k:, k:]
            
            A_inv_B = np.linalg.inv(A_k) @ B_k
        
            abs_of_A_inv_B = np.abs(A_inv_B)
            rho_hat, [i, j] = compute_rho_and_indices(abs_of_A_inv_B, ratio_matrix)

    return R, Phi

# Example usage
M = np.random.rand(5, 5)
f = 2
R, Phi = strong_RRQR(M, f)
print("R:", R)
print("Permutation matrix Phi:", Phi)


In [8]:
def compute_rho_and_indices(A, B):
    max_matrix = np.maximum(A, B)

    # Step 2: Find the maximum value in the max_matrix
    max_value = np.max(max_matrix)
    indices = np.unravel_index(np.argmax(max_matrix), max_matrix.shape)
    
    rho = max_value
    
    return rho, indices

In [9]:
A = np.ones((5, 5))
A

array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.]])

In [10]:
B = 2*np.eye(5)
B

array([[2., 0., 0., 0., 0.],
       [0., 2., 0., 0., 0.],
       [0., 0., 2., 0., 0.],
       [0., 0., 0., 2., 0.],
       [0., 0., 0., 0., 2.]])

In [14]:
np.maximum(A, B)

array([[2., 1., 1., 1., 1.],
       [1., 2., 1., 1., 1.],
       [1., 1., 2., 1., 1.],
       [1., 1., 1., 2., 1.],
       [1., 1., 1., 1., 2.]])

In [15]:
rho, [i, j] = compute_rho_and_indices(A, B)

In [16]:
rho

2.0

In [17]:
i

0

In [20]:
import numpy as np

def apply_givens_rotation(R, k):
    """
    Apply Givens rotations to zero out elements below the diagonal in the first column 
    of the (m-k) x (n-k) part of the mxn matrix R.

    Args:
        R (ndarray): The mxn matrix (modified in place).
        k (int): The index determining the (m-k)x(n-k) submatrix from which elements
                 below the diagonal in the first column are zeroed out.

    Returns:
        None: The matrix R is updated in place.
    """
    m, n = R.shape

    for j in range(k + 1, m):
        x = R[k, k]  # Diagonal element
        y = R[j, k]  # Element below the diagonal to be zeroed

        # If y is already 0, no need to apply a Givens rotation
        if y != 0:
            # Compute Givens rotation matrix
            G = givens(x, y)
            
            # Apply the Givens rotation to the rows k and j of the matrix R
            R[[k, j], k:] = G @ R[[k, j], k:]

# Example usage:
R = np.random.rand(6, 4)

print("Matrix R before Givens rotations:")
print(R)

apply_givens_rotation(R, 2)

print("Matrix R after Givens rotations:")
print(np.round(R, 8))


Matrix R before Givens rotations:
[[0.37454012 0.95071431 0.73199394 0.59865848]
 [0.15601864 0.15599452 0.05808361 0.86617615]
 [0.60111501 0.70807258 0.02058449 0.96990985]
 [0.83244264 0.21233911 0.18182497 0.18340451]
 [0.30424224 0.52475643 0.43194502 0.29122914]
 [0.61185289 0.13949386 0.29214465 0.36636184]]
Matrix R after Givens rotations:
[[ 0.37454012  0.95071431  0.73199394  0.59865848]
 [ 0.15601864  0.15599452  0.05808361  0.86617615]
 [ 0.60111501  0.70807258  0.55263825  0.51776773]
 [ 0.83244264  0.21233911 -0.         -0.94312196]
 [ 0.30424224  0.52475643 -0.         -0.15466664]
 [ 0.61185289  0.13949386 -0.          0.10914907]]


In [72]:
import numpy as np

def apply_givens_rotation_to_submatrix_column_zeroing_except_first(R, k, j):
    """
    Zero out all elements in the j-th column of the (m-k)x(n-k) submatrix of R,
    except for the first element in that column.

    Args:
        R (ndarray): The mxn matrix (modified in place).
        k (int): Index of the start of the submatrix (bottom-right block of size (m-k)x(n-k)).
        j (int): The column index (relative to the submatrix) to be modified.

    Returns:
        None: The matrix R is updated in place.
    """
    m, n = R.shape

    # Actual column index in the full matrix R
    col = k + j

    # Iterate through the rows below the first element in the specified column
    for i in range(col + 1, m):
        # Set the elements below the first element in the column to zero
        R[i, col] = 0



# Example usage:
np.random.seed(seed=42)
R = np.random.rand(8, 6)

print("Matrix R before Givens rotations (on 2nd column of (m-k)x(n-k) block):")
print(R)

# Let's assume k = 0 (i.e., no trimming), and we want to zero out the 2nd column (j = 1 relative to the submatrix)
apply_givens_rotation_to_submatrix_column(R, k=2, j=2)

print("Matrix R after Givens rotations (on 2nd column of (m-k)x(n-k) block):")
print(np.round(R, 8))

Matrix R before Givens rotations (on 2nd column of (m-k)x(n-k) block):
[[0.37454012 0.95071431 0.73199394 0.59865848 0.15601864 0.15599452]
 [0.05808361 0.86617615 0.60111501 0.70807258 0.02058449 0.96990985]
 [0.83244264 0.21233911 0.18182497 0.18340451 0.30424224 0.52475643]
 [0.43194502 0.29122914 0.61185289 0.13949386 0.29214465 0.36636184]
 [0.45606998 0.78517596 0.19967378 0.51423444 0.59241457 0.04645041]
 [0.60754485 0.17052412 0.06505159 0.94888554 0.96563203 0.80839735]
 [0.30461377 0.09767211 0.68423303 0.44015249 0.12203823 0.49517691]
 [0.03438852 0.9093204  0.25877998 0.66252228 0.31171108 0.52006802]]
Matrix R after Givens rotations (on 2nd column of (m-k)x(n-k) block):
[[ 0.37454012  0.95071431  0.73199394  0.59865848  0.15601864  0.15599452]
 [ 0.05808361  0.86617615  0.60111501  0.70807258  0.02058449  0.96990985]
 [ 0.83244264  0.21233911  0.18182497  0.18340451  0.30424224  0.52475643]
 [ 0.43194502  0.29122914  0.61185289  0.13949386  0.29214465  0.36636184]
 [ 0.4

In [112]:
def householder_jth_column(A, j):
    """
    Perform Householder transformation to zero out the elements below the diagonal in the j-th column of matrix A.
    :param A: The input matrix (m x n)
    :param j: The column index (0-based) to apply the Householder transformation
    :return: The transformed matrix with zeroes below the diagonal in the j-th column.
    """
    
    # Get the shape of the matrix
    m, n = A.shape
    R = A.copy()

    # Extract the vector x from the j-th column (from row j to the end)
    x = R[j:, j]

    # Compute the norm of x
    norm_x = np.linalg.norm(x)

    # Create the Householder reflection vector v
    v = x.copy()
    v[0] += np.sign(x[0]) * norm_x

    # Normalize the vector v
    v = v / np.linalg.norm(v)

    # Update the matrix R by applying the Householder transformation
    R[j:, :] -= 2 * np.outer(v, np.dot(v, R[j:, :]))

    return R

In [300]:
np.random.seed(seed=39) #38
R = np.random.rand(8, 6)
k = 1
gamma = np.linalg.norm(R[k:, k:], axis=0) if k < R.shape[0] else np.zeros(R.shape[1] - k)
print(gamma)
print("Matrix R before Givens rotations (on 2nd column of (m-k)x(n-k) block):")
print(R)

[1.18640278 1.18810454 0.70644201 0.75112239 0.78727597]
Matrix R before Givens rotations (on 2nd column of (m-k)x(n-k) block):
[[-2.11799689e+00 -1.65959370e+00 -8.64048690e-01 -1.69091205e+00
  -1.47860917e+00 -1.17904173e+00]
 [ 1.11022302e-16 -5.44074238e-01 -6.33597030e-02 -4.60827535e-01
  -4.42767339e-02  2.40060825e-01]
 [ 2.22044605e-16 -6.23284744e-01  1.14251639e-01 -2.57263907e-02
  -2.50863895e-01 -4.67783776e-01]
 [ 0.00000000e+00 -7.73717943e-02 -5.94331427e-01 -2.61590745e-01
  -7.37441470e-02 -3.54482746e-01]
 [ 2.77555756e-17  6.71875210e-01  4.84322019e-01  3.28418185e-01
   3.80141299e-01  6.80176381e-02]
 [ 2.22044605e-16 -4.74266399e-01 -5.44784378e-01 -2.61114446e-01
  -3.23172460e-01 -5.70227590e-02]
 [ 1.11022302e-16 -1.11201599e-01 -6.23996095e-01 -1.75647326e-01
  -3.52427450e-01 -1.67707773e-01]
 [ 1.11022302e-16  1.68385888e-01 -3.47225728e-01  1.03515648e-01
  -3.47419882e-01 -4.26251904e-01]]


In [301]:
jmax = np.argmax(gamma)
P = np.eye(R.shape[1])
R[:, [k, k + jmax]] = R[:, [k + jmax, k]]
P[[k, k + jmax]] = P[[k + jmax, k]]
print(R)

[[-2.11799689e+00 -8.64048690e-01 -1.65959370e+00 -1.69091205e+00
  -1.47860917e+00 -1.17904173e+00]
 [ 1.11022302e-16 -6.33597030e-02 -5.44074238e-01 -4.60827535e-01
  -4.42767339e-02  2.40060825e-01]
 [ 2.22044605e-16  1.14251639e-01 -6.23284744e-01 -2.57263907e-02
  -2.50863895e-01 -4.67783776e-01]
 [ 0.00000000e+00 -5.94331427e-01 -7.73717943e-02 -2.61590745e-01
  -7.37441470e-02 -3.54482746e-01]
 [ 2.77555756e-17  4.84322019e-01  6.71875210e-01  3.28418185e-01
   3.80141299e-01  6.80176381e-02]
 [ 2.22044605e-16 -5.44784378e-01 -4.74266399e-01 -2.61114446e-01
  -3.23172460e-01 -5.70227590e-02]
 [ 1.11022302e-16 -6.23996095e-01 -1.11201599e-01 -1.75647326e-01
  -3.52427450e-01 -1.67707773e-01]
 [ 1.11022302e-16 -3.47225728e-01  1.68385888e-01  1.03515648e-01
  -3.47419882e-01 -4.26251904e-01]]


In [302]:
print(P)

[[1. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1.]]


In [303]:
if k == 0:
    P_ = np.eye(R.shape[1])
P_ = P*P_

In [304]:
gamma_max = gamma[jmax]
gamma_max

1.1881045421649952

In [305]:
R[k:, k:] = householder_first_column(R[k:, k:])

print("Matrix R after Givens rotations (on 2nd column of (m-k)x(n-k) block):")
print(np.round(R, 5))

Matrix R after Givens rotations (on 2nd column of (m-k)x(n-k) block):
[[-2.118   -0.86405 -1.65959 -1.69091 -1.47861 -1.17904]
 [ 0.       1.1881   0.50833  0.46856  0.6049   0.38607]
 [ 0.      -0.      -0.71936 -0.11057 -0.31013 -0.48111]
 [ 0.       0.       0.42242  0.17978  0.23456 -0.28514]
 [ 0.      -0.       0.26459 -0.03126  0.12891  0.01151]
 [ 0.       0.      -0.01614  0.14347 -0.04057  0.00654]
 [ 0.       0.       0.41354  0.28776 -0.02874 -0.09491]
 [ 0.       0.       0.46038  0.36138 -0.1673  -0.38574]]


In [306]:
print(np.round(R, 3))

[[-2.118 -0.864 -1.66  -1.691 -1.479 -1.179]
 [ 0.     1.188  0.508  0.469  0.605  0.386]
 [ 0.    -0.    -0.719 -0.111 -0.31  -0.481]
 [ 0.     0.     0.422  0.18   0.235 -0.285]
 [ 0.    -0.     0.265 -0.031  0.129  0.012]
 [ 0.     0.    -0.016  0.143 -0.041  0.007]
 [ 0.     0.     0.414  0.288 -0.029 -0.095]
 [ 0.     0.     0.46   0.361 -0.167 -0.386]]


In [307]:
R[k, k+1:]

array([0.50832556, 0.46856263, 0.60490367, 0.38606633])

In [114]:
gamma = np.linalg.norm(R[2:, 2:], axis=0) if 2 < R.shape[0] else np.zeros(6 - 2)
gamma

array([1.04153145, 1.9010947 , 1.312638  , 0.91750044])

In [115]:
jmax = 

1

In [116]:
Ck = np.round(R[2:, 2:], 8)
Ck

array([[0.82581196, 0.94626012, 0.62804605, 0.25226979],
       [0.04845243, 0.85479855, 0.72021402, 0.29597344],
       [0.60249114, 0.15714582, 0.52610225, 0.18759724],
       [0.1564259 , 0.93249624, 0.54295325, 0.65255723],
       [0.01355873, 0.84784478, 0.43507188, 0.47745696],
       [0.11325299, 0.6123622 , 0.22135737, 0.03972313]])

In [118]:
A = householder_jth_column(R, 3)

print("Matrix R after Givens rotations (on 2nd column of (m-k)x(n-k) block):")
print(np.round(A, 8))

Matrix R after Givens rotations (on 2nd column of (m-k)x(n-k) block):
[[ 0.54688916  0.79789902  0.82040188  0.12204987  0.60200201  0.52551458]
 [ 0.46390841  0.47144574  0.63271284  0.92566388  0.81550292  0.94444175]
 [ 0.91958514  0.4148308   0.82581196  0.94626012  0.62804605  0.25226979]
 [-1.34335432 -1.56998547 -0.22003642 -1.64886411 -1.03649466 -0.80062346]
 [ 0.36390928  0.69172878  0.58563907 -0.          0.41584002  0.11876783]
 [-0.05051188 -0.35645619  0.05642647 -0.         -0.11133786  0.24412661]
 [ 0.01583686 -0.00408613 -0.07736281 -0.         -0.15982307  0.10610344]
 [ 0.24181653  0.24575083  0.04758423 -0.         -0.20830994 -0.22848972]]


In [81]:
Ck = np.round(R[2:, 2:], 8)
Ck

array([[ 0.82581196,  0.94626012,  0.62804605,  0.25226979],
       [ 0.04845243,  1.64886411,  1.03649466,  0.80062346],
       [ 0.60249114, -0.        ,  0.3872095 ,  0.13099045],
       [ 0.1564259 , -0.        , -0.21756642,  0.20716448],
       [ 0.01355873, -0.        , -0.1610881 ,  0.01046087],
       [ 0.11325299, -0.        , -0.17618105, -0.27745998]])

In [97]:
Bk_1 = np.round(R[:2, 2:], 8)
Bk_1.shape

(2, 4)

In [91]:
b = Bk_1[:, 0]
B = Bk_1[:, 1:]

In [92]:
B

array([[0.12204987, 0.60200201, 0.52551458],
       [0.92566388, 0.81550292, 0.94444175]])

In [98]:
T = np.zeros((Bk_1.shape))
T[:, 0] = b
T[:, 1:] = B
T

array([[0.82040188, 0.12204987, 0.60200201, 0.52551458],
       [0.63271284, 0.92566388, 0.81550292, 0.94444175]])

In [79]:
Ak_1 = np.round(R[:2, :2], 8)
Ak_1

array([[0.54688916, 0.79789902],
       [0.46390841, 0.47144574]])

In [2]:
from srrqr_algo5 import srrqr_algo5
import numpy as np

np.random.seed(seed=39) #38
R = np.random.rand(8, 6)

ModuleNotFoundError: No module named 'srrqr_algo5'